# Khám phá dữ liệu (EDA) – H&M Personalized Fashion Recommendations

Notebook này khám phá bộ dữ liệu H&M trước khi đưa vào phân cụm và gợi ý sản phẩm. Các biểu đồ được lưu vào thư mục `figures/` (cạnh notebook) để dùng cho báo cáo và slide.

**Yêu cầu:** đã chạy `process.py` và `train.py` (cần `data/raw/hm/transactions_train.parquet`, `data/processed/*`, `backend/models/cluster_profiles.json`).

**Thời gian chạy:** khoảng 1–2 phút (Polars đọc dữ liệu dạng lười, không nạp cả 31,8 triệu dòng vào RAM).

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Tìm thư mục gốc dự án (nơi có config.py) để import config
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 4.5), "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})
PALETTE = ["#4f46e5", "#06b6d4", "#10b981", "#f59e0b", "#f43f5e", "#a855f7"]

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()

def fmt(n):
    return f"{int(n):,}".replace(",", ".")

tx_all = pl.scan_parquet(config.TRANSACTIONS_RAW_PARQUET)          # toàn bộ 2 năm (31,8 triệu dòng)
clean = pl.scan_parquet(config.TRANSACTIONS_CLEAN_PATH)            # 1 năm gần nhất, đã làm sạch
articles = (pl.read_csv(config.ARTICLES_RAW_PATH, schema_overrides={"article_id": pl.Utf8}, infer_schema_length=0)
              .with_columns(pl.col("article_id").str.zfill(10)))
print("Thư mục gốc dự án:", ROOT)
print("Lưu biểu đồ vào  :", FIG_DIR)

## 1. Tổng quan quy mô dữ liệu

In [ ]:
overview = tx_all.select(
    pl.len().alias("Số giao dịch"),
    pl.col("customer_id").n_unique().alias("Số khách hàng"),
    pl.col("article_id").n_unique().alias("Số sản phẩm có bán"),
    pl.col("t_dat").min().alias("Từ ngày"),
    pl.col("t_dat").max().alias("Đến ngày"),
).collect()
customers_raw = pl.scan_csv(config.CUSTOMERS_RAW_PATH).select(pl.len()).collect().item()
print(f"Sản phẩm trong danh mục (articles.csv): {fmt(len(articles))}")
print(f"Khách trong hồ sơ (customers.csv)    : {fmt(customers_raw)}")
overview

## 2. Số giao dịch theo tháng – tính mùa vụ

Thời trang có tính mùa vụ rõ rệt. Biểu đồ cho thấy lượng giao dịch thay đổi theo mùa, cũng là lý do hệ thống chỉ dùng **1 năm gần nhất** (đủ 4 mùa) và chia train/test theo thời gian.

In [ ]:
monthly = (tx_all.group_by(pl.col("t_dat").dt.truncate("1mo").alias("thang"))
           .agg(pl.len().alias("so_giao_dich")).sort("thang").collect())
start = np.datetime64(config.DATA_START_DATE) if config.DATA_START_DATE else None

fig, ax = plt.subplots()
colors = [PALETTE[0] if (start is None or np.datetime64(d) >= start) else "#cbd5e1" for d in monthly["thang"]]
ax.bar(monthly["thang"].to_list(), monthly["so_giao_dich"].to_list(), width=25, color=colors)
ax.set_title("Số giao dịch theo tháng (màu đậm = dữ liệu được dùng cho mô hình)")
ax.set_ylabel("Số giao dịch")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M"))
save(fig, "01_giao_dich_theo_thang")

top = monthly.sort("so_giao_dich", descending=True).head(3)
print("3 tháng bán chạy nhất:", [(str(d)[:7], fmt(n)) for d, n in top.iter_rows()])

## 3. Kênh bán hàng: online hay tại cửa hàng

In [ ]:
ch = (clean.group_by("sales_channel_id").agg(pl.len().alias("n")).sort("sales_channel_id").collect())
labels = {1: "Cửa hàng (1)", 2: "Online (2)"}
fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(ch["n"], labels=[labels.get(c, str(c)) for c in ch["sales_channel_id"]], autopct="%1.1f%%",
       colors=PALETTE[:len(ch)], startangle=90, wedgeprops={"width": 0.45})
ax.set_title("Tỷ lệ giao dịch theo kênh bán (1 năm gần nhất)")
save(fig, "02_kenh_ban")

## 4. Phân bố tuổi khách hàng

In [ ]:
cust = pl.read_parquet(config.CUSTOMERS_PATH)
ages = cust["age"].drop_nulls().cast(pl.Float64)
fig, ax = plt.subplots()
ax.hist(ages.to_numpy(), bins=range(15, 100, 2), color=PALETTE[1], edgecolor="white")
ax.set_title("Phân bố tuổi khách hàng")
ax.set_xlabel("Tuổi"); ax.set_ylabel("Số khách")
save(fig, "03_tuoi_khach")
print(f"Có tuổi: {fmt(len(ages))} / {fmt(len(cust))} khách | trung vị {ages.median():.0f} tuổi | "
      f"{(ages.is_between(18, 30)).mean():.1%} khách 18–30 tuổi")

## 5. Nhóm sản phẩm và màu sắc bán chạy

In [ ]:
sold = (clean.group_by("stockcode").agg(pl.len().alias("n")).collect()
        .join(articles.rename({"article_id": "stockcode"}), on="stockcode", how="left"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col, title, color in [(axes[0], "product_group_name", "Top nhóm sản phẩm", PALETTE[0]),
                              (axes[1], "colour_group_name", "Top màu sắc", PALETTE[3])]:
    t = sold.group_by(col).agg(pl.col("n").sum()).sort("n", descending=True).head(10)
    ax.barh(t[col].to_list()[::-1], t["n"].to_list()[::-1], color=color)
    ax.set_title(f"{title} (theo lượt mua)")
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M"))
save(fig, "04_nhom_san_pham_mau")

## 6. Phân bố giá (giá đã được H&M chuẩn hóa)

Giá trong dữ liệu **không phải tiền thật** mà đã được chuẩn hóa, nên chỉ dùng để so sánh tương đối.

In [ ]:
prices = clean.select("price").collect()["price"]
fig, ax = plt.subplots()
ax.hist(prices.to_numpy(), bins=100, range=(0, prices.quantile(0.99)), color=PALETTE[2], edgecolor="white")
ax.set_title("Phân bố giá mỗi món (cắt ở phân vị 99%)")
ax.set_xlabel("Giá (chuẩn hóa)"); ax.set_ylabel("Số giao dịch")
save(fig, "05_phan_bo_gia")
print(f"Giá: trung vị {prices.median():.4f} | trung bình {prices.mean():.4f} | max {prices.max():.4f}")

## 7. "Đuôi dài" sản phẩm

Một số ít sản phẩm chiếm phần lớn lượt mua. Đây là cơ sở để mô hình gợi ý chỉ dùng **10.000 sản phẩm phổ biến nhất** và chỉ lấy ảnh cho **15.000 sản phẩm** mà vẫn phủ phần lớn nhu cầu.

In [ ]:
counts = np.sort(sold["n"].to_numpy())[::-1]
cum = np.cumsum(counts) / counts.sum()
x = np.arange(1, len(counts) + 1)
fig, ax = plt.subplots()
ax.plot(x, cum * 100, color=PALETTE[0], lw=2)
for k, c in [(10_000, PALETTE[4]), (15_000, PALETTE[2])]:
    if k <= len(cum):
        ax.axvline(k, ls="--", color=c)
        ax.annotate(f"Top {fmt(k)}: {cum[k-1]:.1%}", (k, cum[k-1] * 100), xytext=(8, -18),
                    textcoords="offset points", color=c, fontweight="bold")
ax.set_title("Tỷ lệ lượt mua tích lũy theo số sản phẩm phổ biến nhất")
ax.set_xlabel("Số sản phẩm (xếp theo lượt mua giảm dần)"); ax.set_ylabel("% tổng lượt mua")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
save(fig, "06_duoi_dai_san_pham")

## 8. RFM: phân bố gốc và sau biến đổi log

Recency, Frequency, Monetary đều **lệch phải rất mạnh** (đa số khách mua ít, một số rất ít mua rất nhiều). K-Means nhạy với giá trị lớn, nên hệ thống biến đổi `log1p` trước khi chuẩn hóa và phân cụm.

In [ ]:
rfm = pl.read_parquet(config.RFM_TRAIN_PATH)
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for j, col in enumerate(["recency", "frequency", "monetary"]):
    v = rfm[col].to_numpy().astype(float)
    axes[0, j].hist(v, bins=60, color=PALETTE[j], edgecolor="white")
    axes[0, j].set_title(f"{col.capitalize()} – gốc")
    axes[1, j].hist(np.log1p(v), bins=60, color=PALETTE[j], edgecolor="white", alpha=0.8)
    axes[1, j].set_title(f"{col.capitalize()} – sau log1p")
save(fig, "07_rfm_goc_va_log")

## 9. Kết quả phân cụm: quy mô và đặc điểm 4 nhóm khách

In [ ]:
profiles = json.loads(config.CLUSTER_PROFILES_PATH.read_text(encoding="utf-8"))
prof = pl.DataFrame(profiles).sort("size", descending=True)
color_by_name = {"Khách VIP": "#10b981", "Khách trung thành": "#818cf8",
                 "Khách mới": "#06b6d4", "Khách có nguy cơ rời bỏ": "#f43f5e"}
cols = [color_by_name.get(n, "#94a3b8") for n in prof["name"]]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].bar(prof["name"], prof["size"], color=cols); axes[0].set_title("Số khách")
axes[1].bar(prof["name"], prof["mean_recency"], color=cols); axes[1].set_title("Recency TB (ngày)")
axes[2].bar(prof["name"], prof["mean_frequency"], color=cols); axes[2].set_title("Frequency TB (đơn)")
axes[3].bar(prof["name"], prof["mean_monetary"], color=cols); axes[3].set_title("Monetary TB (chuẩn hóa)")
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
save(fig, "08_ho_so_cum")
prof.select("name", "size", "pct", "mean_recency", "mean_frequency", "mean_monetary")

## 10. Chia train/test theo thời gian

Mô hình chỉ học từ dữ liệu **trước** mốc cắt và được chấm điểm trên **28 ngày cuối** (những gì khách thực sự mua sau đó), giống như dùng quá khứ để đoán tương lai.

In [ ]:
daily = clean.group_by(pl.col("invoice_date").dt.date().alias("ngay")).agg(pl.len().alias("n")).sort("ngay").collect()
split = daily["ngay"].max() - np.timedelta64(config.TEST_DAYS, "D")
is_test = daily["ngay"].to_numpy() > np.datetime64(split)
fig, ax = plt.subplots()
ax.plot(daily["ngay"].to_list(), daily["n"].to_list(), color=PALETTE[0], lw=1)
ax.fill_between(daily["ngay"].to_list(), 0, daily["n"].to_list(), where=is_test, color=PALETTE[4], alpha=0.35,
                label=f"Tập test ({config.TEST_DAYS} ngày cuối)")
ax.axvline(split, color=PALETTE[4], ls="--")
ax.set_title("Số dòng giao dịch mỗi ngày và mốc chia train/test")
ax.legend(loc="upper left")
save(fig, "09_chia_train_test")

## Tóm tắt

- Dữ liệu lớn (31,8 triệu giao dịch, gần 1,4 triệu khách) nhưng **rất thưa**: mỗi khách chỉ mua một phần rất nhỏ trong hàng chục nghìn sản phẩm.
- Giao dịch **có tính mùa vụ** → chia train/test theo thời gian và dùng dữ liệu gần đây.
- Lượt mua tập trung vào **một số ít sản phẩm** (đuôi dài) → giới hạn mô hình ở các sản phẩm phổ biến mà vẫn phủ phần lớn nhu cầu.
- RFM **lệch phải mạnh** → cần biến đổi log trước khi phân cụm.

Các biểu đồ đã được lưu trong thư mục `figures/`.